[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session12/PHI_Detection_HuggingFace.ipynb)

# PHI Detection with a Hugging Face Model
## CCI Session 12

**Duration:** ~30 minutes &nbsp;|&nbsp; **Runtime:** CPU is fine (model is only 66M params)

### Clinical Scenario
> Before any clinical free-text can leave the hospital for research or a cloud LLM, it must be screened for **Protected Health Information (PHI)**. In this lab you load a ready-made PHI detector from the Hugging Face Hub and run it over **synthetic** clinical notes to find — and redact — the 18 categories of identifiers named in the HIPAA Safe Harbor rule.

### Objectives
- Load [`mkocher/hipaa-phi-detector`](https://huggingface.co/mkocher/hipaa-phi-detector) with the `transformers` `pipeline` API.
- Inspect the PHI entity categories the model was trained to detect.
- Run named-entity recognition (NER) over synthetic notes and read the output.
- Build a **redaction** function that replaces each detected span with its category tag.
- Summarise PHI findings across a small corpus and discuss limitations.

> ⚠️ **Every note in this notebook is fully synthetic.** Names, MRNs, dates, phone numbers, and IDs were invented for teaching and refer to no real person.

## About the model

`mkocher/hipaa-phi-detector` is a **token-classification (NER)** model:

| | |
|---|---|
| **Base** | DistilBERT (~66M parameters) |
| **Task** | Token classification with a BIO tagging head |
| **Training** | 5,000+ synthetic HIPAA examples, 37 BIO labels → 18 PHI categories |
| **License** | Apache 2.0 |

It targets the **18 HIPAA Safe Harbor identifiers** — names, geographic detail, dates, phone/fax, email, SSN, medical-record and health-plan numbers, account/license/vehicle/device IDs, URLs, IP addresses, biometric and photographic identifiers.

## 1. Setup

Install `transformers` and a backend (`torch`). On Google Colab `torch` is usually pre-installed, so this is quick. No API key and no GPU are required.

In [ ]:
!pip install -q transformers torch pandas

## 2. Load the model from the Hugging Face Hub

The `pipeline` helper downloads the model + tokenizer on first use and caches them.
`aggregation_strategy="simple"` merges word-piece tokens back into whole words and
groups consecutive tokens of the same type into a single entity span.

In [ ]:
from transformers import pipeline
import pandas as pd

MODEL_ID = "mkocher/hipaa-phi-detector"

phi_detector = pipeline(
    task="token-classification",
    model=MODEL_ID,
    aggregation_strategy="simple",   # merge sub-word tokens into clean spans
)

print("Loaded:", MODEL_ID)

### What can it detect?

The model's config carries the full label set. With BIO tagging each PHI category
appears twice — `B-` (beginning of an entity) and `I-` (inside/continuation) — plus a
single `O` (outside / not PHI).

In [ ]:
id2label = phi_detector.model.config.id2label

# Strip the B-/I- prefixes to see the distinct PHI categories
categories = sorted({lab.split("-", 1)[-1] for lab in id2label.values() if lab != "O"})

print(f"{len(id2label)} BIO labels covering {len(categories)} PHI categories:\n")
for c in categories:
    print("  •", c)

## 3. Synthetic clinical notes

A small corpus of **invented** notes. Between them they include names, ages (incl. >89,
which HIPAA treats specially), geographic detail, dates, phone/fax numbers, email, SSN,
medical-record and health-plan numbers, vehicle and device IDs, a URL, and an IP address.

In [ ]:
SYNTHETIC_NOTES = [
    # Note 0 — pediatric oncology clinic note
    """Patient: Lana Al-Masri
MRN: 04417829   DOB: 03/14/2015   Visit date: 06/02/2026

Lana is an 11-year-old girl from Irbid, Jordan, followed for B-cell acute
lymphoblastic leukemia diagnosed on 01/20/2024. She presented today to the
King Hussein Cancer Center pediatric oncology clinic with her mother,
Mrs. Huda Al-Masri, reachable at +962 79 123 4567.

Referring physician Dr. Omar Haddad emailed records to
o.haddad@example-hospital.jo. The family lives at 22 Rainbow Street, Amman.
Health plan ID: HPJ-99812. SSN on file: 521-88-7634.
Plan: continue maintenance chemotherapy; next clinic 07/14/2026.""",

    # Note 1 — adult inpatient discharge (note the age > 89)
    """Patient Name: Robert J. Henderson
Medical Record #: 7781204   Age: 92   Admitted: May 3, 2026 to Ward 5B.

Mr. Henderson is a 92-year-old male resident of Springfield who underwent
resection for stage III colon adenocarcinoma. Home phone (217) 555-0148.
Emergency contact: his daughter Sarah Henderson, sarah.h@email.com.
Vehicle plate ABC-1234 recorded for the parking pass.
Discharge summary faxed to 217-555-0199 on 05/10/2026.""",

    # Note 2 — tele-oncology consult
    """Consult note — 06/15/2026
Re: Mariam Khalil, 7 yo F, MRN 0098231. Telemonitoring device serial
DEV-55102 transmitted from home IP address 192.168.1.45. Mother's mobile
0790 555 222. Seen at our Zarqa satellite clinic. Patient portal:
https://patients.example.jo/mariam. Diagnosis: Wilms tumor, right kidney.""",
]

print(f"{len(SYNTHETIC_NOTES)} synthetic notes loaded.\n")
print(SYNTHETIC_NOTES[0])

## 4. Detect PHI in a single note

Calling the pipeline on a string returns a list of entity dicts. With
`aggregation_strategy="simple"` each dict has:
`entity_group`, `score`, `word`, `start`, `end` (character offsets).

In [ ]:
note = SYNTHETIC_NOTES[0]
entities = phi_detector(note)

print(f"{len(entities)} PHI spans found\n")
print(f"{'CATEGORY':<16}{'SCORE':<8}{'TEXT'}")
print("-" * 50)
for e in entities:
    # Use the character offsets to pull the exact original text
    span_text = note[e["start"]:e["end"]]
    print(f"{e['entity_group']:<16}{e['score']:<8.2f}{span_text!r}")

Putting the same result into a `DataFrame` makes it easy to sort, filter, and export.

In [ ]:
def entities_to_df(text, entities):
    rows = [
        {
            "category": e["entity_group"],
            "text": text[e["start"]:e["end"]],
            "score": round(float(e["score"]), 3),
            "start": e["start"],
            "end": e["end"],
        }
        for e in entities
    ]
    return pd.DataFrame(rows, columns=["category", "text", "score", "start", "end"])

entities_to_df(note, entities)

## 5. Redact the note

To de-identify, replace each detected span with a `[CATEGORY]` tag. We sort spans by
start offset **descending** so that replacing later spans never shifts the offsets of
earlier ones. A confidence threshold lets you trade recall for precision.

In [ ]:
def redact(text, entities, min_score=0.5):
    """Return `text` with each detected PHI span replaced by [CATEGORY]."""
    spans = sorted(
        (e for e in entities if e["score"] >= min_score),
        key=lambda e: e["start"],
        reverse=True,
    )
    out = text
    for e in spans:
        out = out[:e["start"]] + f"[{e['entity_group']}]" + out[e["end"]:]
    return out

print(redact(note, entities))

## 6. Run across the whole corpus

Detect, redact, and summarise every note in one pass.

In [ ]:
for i, n in enumerate(SYNTHETIC_NOTES):
    ents = phi_detector(n)
    print("=" * 70)
    print(f"NOTE {i}  —  {len(ents)} PHI spans detected")
    print("=" * 70)
    print(redact(n, ents))
    print()

### PHI category counts across the corpus

A quick audit: which identifier types show up, and how often?

In [ ]:
all_rows = []
for i, n in enumerate(SYNTHETIC_NOTES):
    for e in phi_detector(n):
        all_rows.append({"note": i, "category": e["entity_group"]})

corpus_df = pd.DataFrame(all_rows)
summary = (
    corpus_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)
summary

## 7. Your turn

Try a few experiments:

1. **Add your own synthetic note.** Append a fictional note to `SYNTHETIC_NOTES`, rerun the corpus loop, and check what the model catches — and misses.
2. **Tune the threshold.** Call `redact(note, entities, min_score=0.9)`. Which true identifiers slip through at high confidence? This is the recall/precision trade-off that matters clinically.
3. **Probe a hard case.** Feed the model an unusual identifier format (e.g. an international phone number, a non-Latin name, a relative date like "last Tuesday") and see whether it generalises.

In [ ]:
# Scratch space for your experiments
my_note = """..."""   # paste a synthetic note here
# print(redact(my_note, phi_detector(my_note)))

## 8. Limitations & safe use

- **A detector is not a guarantee.** NER models miss identifiers (false negatives) and over-tag (false positives). De-identification for real data needs human review and validation against a gold standard.
- **Trained on synthetic data.** Performance on your real institutional notes — different templates, Arabic/English code-switching, local ID formats — will differ and should be measured before any reliance.
- **Safe Harbor is more than NER.** True HIPAA Safe Harbor de-identification also requires handling dates (shifting, not just removing), ages > 89, and small-population geographic units — policy decisions beyond what a tagger outputs.
- **Never paste real PHI into a public Colab or a third-party LLM.** Run de-identification inside an approved environment first.

### Recap
You loaded a Hugging Face token-classification model, inspected its PHI categories, ran it over synthetic notes, redacted detected spans, and audited findings across a corpus — the core loop of an automated PHI-screening step.